# Checking the Assumptions of Linear Regression

In notebook 01 you listed the assumptions linear regression makes, and in notebook 06 you saw that the p-values in a statsmodels summary rely on them. This notebook shows how to check each assumption in practice, on the cars data, and what to do when a check fails.

> [!TIP]
> This notebook is optional. Work through it if you have time left, or come back to it when you build regression models in the weekly project.

## Learning Objectives

At the end of this notebook, you should be able to:

- Plot the residuals against the fitted values to check linearity and constant variance.
- Run a Breusch-Pagan test and calculate variance inflation factors to back up what the plots show.
- Identify an omitted variable by correlating it with the residuals.
- Compare the diagnostics of a model before and after a log transformation of the target.

## The Model to Check

We check a simple model: `mpg` explained by `weight`, on all 392 cars of the dataset from notebook 03. (The small `cars.csv` from notebook 01 has only 32 cars, too few for patterns in the residuals to show.)

As in notebook 06, we fit the models with `sm.OLS` and add the constant for the intercept ourselves.

Documentation: [`OLS`](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html), [`add_constant`](https://www.statsmodels.org/stable/generated/statsmodels.tools.tools.add_constant.html).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

cars = pd.read_csv("data/cars_multivariate.csv")

model = sm.OLS(cars["mpg"], sm.add_constant(cars[["weight"]])).fit()
print(f"R-squared: {model.rsquared:.3f}")

The model explains 69.3% of the variance in `mpg`. That sounds good, but $R^2$ says nothing about whether the assumptions behind the model hold. The residuals do.

| Assumption | How to check it |
|---|---|
| Linearity | Plot the residuals against the fitted values: look for a curve |
| Constant variance (homoscedasticity) | The same plot: look for a funnel, then run a Breusch-Pagan test |
| Zero-mean error | Nothing to check when the model has an intercept |
| Strict exogeneity | No test: think about what the model leaves out, and correlate it with the residuals |
| No multicollinearity | Variance inflation factors (VIF) for the features |
| Normal residuals (needed for p-values) | A Q-Q plot |

## Linearity and Constant Variance

Both assumptions are checked with one plot: the residuals against the fitted values. If the model is right, the points form a horizontal band around zero with no shape.

- A **curve** means the relationship is not linear.
- A **funnel**, where the spread grows or shrinks, means the variance is not constant (**heteroscedasticity**).

In [ ]:
def plot_residuals(model, title):
    plt.figure(figsize=(7, 4))
    plt.scatter(model.fittedvalues, model.resid, s=12, alpha=0.6)
    plt.axhline(0, color="magenta", linewidth=2)
    plt.xlabel("fitted values")
    plt.ylabel("residuals")
    plt.title(title)
    plt.show()


plot_residuals(model, "mpg explained by weight")

The plot shows both problems.

- The points bend in a **U**: most residuals are positive for the lowest and the highest fitted values and negative in the middle, so the model underestimates `mpg` at both ends.
- The band **widens** from left to right.

To put a number on the funnel, split the cars into three groups by fitted value and compare the spread of the residuals:

In [ ]:
groups = pd.qcut(model.fittedvalues, 3, labels=["low", "middle", "high"])
model.resid.groupby(groups, observed=True).std().round(2)

The standard deviation of the residuals doubles, from 2.67 mpg for the cars with the lowest fitted values to 5.35 mpg for those with the highest.

### A Formal Test: Breusch-Pagan

The **Breusch-Pagan test** checks the same thing with a hypothesis test. Its null hypothesis is that the variance of the residuals is constant, so a small p-value is evidence of heteroscedasticity.

Documentation: [`het_breuschpagan`](https://www.statsmodels.org/stable/generated/statsmodels.stats.diagnostic.het_breuschpagan.html).

In [ ]:
bp_pvalue = het_breuschpagan(model.resid, model.model.exog)[1]
print(f"Breusch-Pagan p-value: {bp_pvalue:.2g}")

A p-value of 2.1e-06 is far below 0.05: the variance is not constant. The standard errors and p-values in this model's summary are therefore unreliable.

## Zero-Mean Error

In [ ]:
print(f"mean of the residuals: {model.resid.mean():.1e}")

The mean is zero up to rounding error. This is not a finding about our data: whenever a model has an intercept, ordinary least squares makes the residuals average exactly zero. There is nothing to check.

## Strict Exogeneity

Strict exogeneity means the residuals are unrelated to the features, and to anything else that drives the target. It cannot be tested from the residuals alone, because ordinary least squares makes them uncorrelated with the features in the model by construction.

What you can do is think about what the model leaves out, and check whether it correlates with the residuals. Newer cars are more economical, and `model_year` is not in our model:

In [ ]:
corr = model.resid.corr(cars["model_year"])
print(f"correlation between the residuals and model_year: {corr:.2f}")

A correlation of 0.58 means the residuals carry information the model misses: for a given weight, newer cars do better than the model predicts.

This only biases the model if the omitted variable is also related to a feature in the model. Newer cars in this dataset are lighter (the correlation between `weight` and `model_year` is -0.31), so part of the effect of `model_year` has been credited to `weight`. Adding `model_year` shows how much:

In [ ]:
model_with_year = sm.OLS(cars["mpg"], sm.add_constant(cars[["weight", "model_year"]])).fit()
print(f"weight coefficient without model_year: {model.params['weight']:.5f}")
print(f"weight coefficient with model_year:    {model_with_year.params['weight']:.5f}")

The coefficient of `weight` shrinks from -0.00765 to -0.00663: without `model_year`, the model overstated the effect of `weight` on `mpg` by about 15%. The fix for a missing variable is to add it to the model.

## No Multicollinearity

With several features, check that none of them is almost a linear combination of the others. The **variance inflation factor (VIF)** measures how much the variance of a coefficient grows because of this. A common rule of thumb treats a VIF above 5 as a warning and above 10 as a problem.

Here are the four numeric features from notebook 03:

Documentation: [`variance_inflation_factor`](https://www.statsmodels.org/stable/generated/statsmodels.stats.outliers_influence.variance_inflation_factor.html).

In [ ]:
X = sm.add_constant(cars[["displacement", "horsepower", "weight", "acceleration"]])

# variance_inflation_factor is one of the few functions that needs a NumPy array (.values)

vif = pd.Series(
    [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])],
    index=X.columns[1:],
)
vif.round(1)

`displacement` (10.7), `weight` (10.3) and `horsepower` (8.8) are strongly related: heavy cars tend to have big, powerful engines. Their individual coefficients are unstable. The overlap also explains why adding `displacement` and `acceleration` in notebook 03 barely changed $R^2$: the new features carried little information the model did not already have. When you need coefficients you can interpret, keep only one of the related features.

## Normal Residuals

The Gauss-Markov theorem does not need normally distributed residuals, but the p-values and confidence intervals in a statsmodels summary do. A **Q-Q plot** compares the residuals with a normal distribution: if they are normal, the points lie on the straight line.

Documentation: [`qqplot`](https://www.statsmodels.org/stable/generated/statsmodels.graphics.gofplots.qqplot.html).

In [ ]:
sm.qqplot(model.resid, line="s")
plt.title("Q-Q plot: mpg explained by weight")
plt.show()

The points follow the line in the middle but rise above it at the right end: there are more large positive residuals than a normal distribution would produce (the residuals are skewed to the right). These are the economical cars the model underestimates.

## The Fix: Model the Log of mpg

The curve and the funnel have the same cause: each extra unit of weight changes `mpg` by a similar *percentage*, not by a similar *amount*. Modelling the logarithm of `mpg` turns those percentage changes into constant ones. The only change is the target: `np.log(cars["mpg"])` instead of `cars["mpg"]`.

In [ ]:
log_model = sm.OLS(np.log(cars["mpg"]), sm.add_constant(cars[["weight"]])).fit()

print(f"R-squared: {log_model.rsquared:.3f}")
print(f"Breusch-Pagan p-value: {het_breuschpagan(log_model.resid, log_model.model.exog)[1]:.2f}")

plot_residuals(log_model, "log(mpg) explained by weight")

sm.qqplot(log_model.resid, line="s")
plt.title("Q-Q plot: log(mpg) explained by weight")
plt.show()

One change fixes all three problems, the curve, the funnel and the skewed residuals:

| Check | `mpg` explained by `weight` | log of `mpg` explained by `weight` |
|---|---|---|
| Residual plot | U-shaped curve and a funnel | Even band, no clear curve |
| Residual spread, low to high fitted values | 2.67 to 5.35 | 0.14 to 0.17 |
| Breusch-Pagan p-value | 2.1e-06 | 0.10 |
| Q-Q plot | Right tail above the line | Close to the line |
| $R^2$ | 0.693 | 0.767 |

The two $R^2$ values are measured on different scales (`mpg` and its logarithm), so compare them with care. The real improvement is in the diagnostics.

The coefficient now reads as a percentage change. The `weight` coefficient is -0.000351, so each extra 1,000 lb lowers `mpg` by about 30% ($e^{-0.351} - 1 \approx -0.30$).

## Try It

Repeat the checks for a model of `mpg` explained by `horsepower`: plot the residuals, compare their spread, and run the Breusch-Pagan test. Then try the log fix. Does the same problem appear, and does the fix work as well?

In [ ]:
# TODO: fit mpg explained by horsepower (remember the constant)
hp_model = ...

# The same checks as for weight
plot_residuals(hp_model, "mpg explained by horsepower")
groups = pd.qcut(hp_model.fittedvalues, 3, labels=["low", "middle", "high"])
print(hp_model.resid.groupby(groups, observed=True).std().round(2))
# TODO: pass the residuals and the model's exog, as above
print(f"Breusch-Pagan p-value: {het_breuschpagan(..., ...)[1]:.2g}")

# TODO: the log fix: fit the log of mpg explained by horsepower, then repeat the checks
log_hp_model = ...

<details><summary>Show answer</summary>

The same two problems appear: a U-shaped curve and a funnel, with the residual spread growing from 3.85 to 5.39 mpg (Breusch-Pagan p = 0.0031).

With the log of `mpg` as the target, the spread evens out (0.18 to 0.19) and the Breusch-Pagan p-value rises to 0.43, so the variance is fixed. A slight curve remains at the highest fitted values, so the relationship is not fully linear yet. Transforming `horsepower` as well, or adding a squared term, is worth trying.

</details>

## Summary

In this notebook you:

- Plotted the residuals of the model of `mpg` explained by `weight` against the fitted values and found a curve (non-linearity) and a funnel (heteroscedasticity).
- Ran a Breusch-Pagan test (p = 2.1e-06) and calculated VIFs above 8 for `displacement`, `weight` and `horsepower`.
- Identified `model_year` as an omitted variable: it correlates 0.58 with the residuals and biases the `weight` coefficient.
- Compared the diagnostics before and after modelling the log of `mpg`, which removed the curve and the funnel.

Two assumptions need no test: zero-mean error holds automatically when the model has an intercept, and strict exogeneity cannot be tested, only reasoned about by asking what the model leaves out.

## References & Further Reading

- [**statsmodels regression diagnostics example**](https://www.statsmodels.org/stable/examples/notebooks/generated/regression_diagnostics.html): the official walkthrough of the tests used here, and more.
- [**statsmodels diagnostic tests**](https://www.statsmodels.org/stable/diagnostic.html): the full list of residual and specification tests, including Breusch-Pagan.
- [**Variance inflation factor (statsmodels)**](https://www.statsmodels.org/stable/generated/statsmodels.stats.outliers_influence.variance_inflation_factor.html): the function used for the multicollinearity check.
- [**An Introduction to Statistical Learning**](https://www.statlearning.com/): section 3.3.3, "Potential Problems", explains non-linearity, heteroscedasticity and collinearity with residual plots.